In [1]:
import pandas as pd
import gzip
import numpy as np
from pathlib import Path

In [1]:
from torch import nn
import torch

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        dummy = torch.empty(0)
        self.register_buffer("_device_anchor", dummy, persistent=False)

    @property
    def device(self):
        return self._device_anchor.device

In [3]:
m = Model()
m.device

device(type='cpu')

In [6]:
m = Model().to("cuda:1")
m.device

device(type='cuda', index=1)

In [2]:
datasets = ["beauty", "ml1m", "clothes", "sports"]
models = ["sasrec", "mrgsrec"]

print("ndcg@10\t\tsasrec\t\t\tmrgsrec")
for dataset in datasets:
    print(dataset, end="\t\t")
    for model in models:
        p = Path(f"optuna_outputs/{dataset}_{model}")
        df = pd.read_csv(p / "index.csv")
        print(df["ndcg@10"].max(), end="\t")
    print()

ndcg@10		sasrec			mrgsrec
beauty		0.0540641284194715	0.0294226747820267	
ml1m		0.0646593084151776	0.0476194916823447	
clothes		0.0097791986250019	0.009984695212562	
sports		0.0189404549721057	0.0213774554554924	


In [2]:
def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield eval(l)

def getDF(path):
  i = 0
  df = []
  for d in parse(path):
    df.append(d)
  return pd.DataFrame(df)

df = getDF('raw_data/reviews_Kindle_Store_5.json.gz')

In [24]:
df = pd.read_csv("raw_data/beer.csv")
df = df.rename(columns={"userid": "user_id", "itemid": "item_id"})
df["rating"] = 1
new_df = df
new_df["user_id"], unique_user_ids = pd.factorize(new_df["user_id"])
new_df["item_id"], unique_item_ids = pd.factorize(new_df["item_id"])
new_df["user_id"] += 1
new_df["item_id"] += 1
new_df = new_df.reset_index()
df_sorted = new_df.sort_values(by=["timestamp", "index"])
df_sorted = df_sorted.drop(columns=["index"])
df_sorted = df_sorted.reset_index(drop=True)

In [25]:
print(df_sorted.user_id.max(), df_sorted.user_id.min(), df_sorted.user_id.nunique())
print(df_sorted.item_id.max(), df_sorted.item_id.min(), df_sorted.item_id.nunique())

7606 1 7606
21613 1 21613


In [3]:
new_df = df[["reviewerID", "asin", "overall", "unixReviewTime"]].copy()
new_df.columns = ["user_id", "item_id", "rating", "timestamp"]
new_df["rating"] = 1

In [4]:
new_df["user_id"], unique_user_ids = pd.factorize(new_df["user_id"])
new_df["item_id"], unique_item_ids = pd.factorize(new_df["item_id"])
new_df["user_id"] += 1
new_df["item_id"] += 1

In [5]:
new_df = new_df.reset_index()

In [7]:
df_sorted = new_df.sort_values(by=["timestamp", "index"])
df_sorted = df_sorted.drop(columns=["index"])
df_sorted = df_sorted.reset_index(drop=True)

In [26]:
df_sorted.to_csv("beer.csv", index=False)

In [16]:
test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby("user_id")["timestamp"].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby("user_id")["timestamp"].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]


train = pd.concat([train, warm_v])

In [17]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print(
    "test_val intersection users: ",
    np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0],
)

Train len:  192817
Train users:  22236
Val len:  1714
Val users:  1714
Test len:  1673
Test users:  1673
Warm val len:  2256
Warm val users:  808
Warm test len:  2298
Warm test users:  772
test_val intersection users:  428


In [19]:
dataset_folder = Path("new_data/Beauty")
dataset_folder.mkdir(parents=True)

In [20]:
user_items = train.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "train.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [21]:
train_warm_v = pd.concat([train, warm_v], ignore_index=True)
user_items = warm_v.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "warm_val.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [22]:
filtered_train = train_warm_v[train_warm_v["user_id"].isin(val["user_id"])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby("user_id")["item_id"].apply(list).to_dict()
with open(dataset_folder / "val.txt", "w") as f:
    for user_id, items in user_items.items():
        if len(items) > 1:
            f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [23]:
unique_indices = val["user_id"].unique()
with open(dataset_folder / "val_users.txt", "w") as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open(dataset_folder / "val_users.txt", "r") as f:
    index_list = [int(line.strip()) for line in f]

In [24]:
filtered_train = df_sorted[df_sorted["user_id"].isin(test["user_id"])]


user_items = filtered_train.groupby("user_id")["item_id"].apply(list).to_dict()
with open(dataset_folder / "test.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [25]:
user_items = df_sorted.groupby("user_id")["item_id"].apply(list).to_dict()

with open(dataset_folder / "all_data.txt", "w") as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")